# Chapter 3 — Sorting algorithms (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch03_sorting_cpp.ipynb)

Executable notebook of the chapter “Sorting algorithms”: it contains the algorithms `InsertionSort`, `ExchangeSort` and `SelectionSort` written in C++ and run on the same examples as the lecture notes, printing the results and, where the notes show a trace, the step-by-step trace in the same format.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/algorithms/basic-algorithms/03-sorting/). The same notebook is also available in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch03_sorting_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch03_sorting_java.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`. Each C++ program is written to a file by the `%%writefile name.cpp` cell, compiled with `g++ -std=c++17 -O2` and run by the next cell; nothing to install (Colab already has `g++`). The programs read their input from the command line: the “Try it yourself” cell runs the same executable with numbers you can change.

**Indices.** In the pseudo-code arrays start from 1, here from 0: the element $a[i]$ of the pseudo-code is `a[i-1]` in the code. The printed traces use the indices of the code (as in the C++ examples of the lecture notes), so they are those of the pseudo-code with every index decreased by 1. The function names are those of the pseudo-code, as in the lecture notes (`FindMaximum`).

## InsertionSort

**Input:** an array $a[1..n]$ of $n$ natural numbers. **Output:** the array $a$ sorted in non-decreasing order.

At step $i$ the sub-array $a[1..i-1]$ is sorted: the key $a[i]$ is inserted in its place, after shifting one position to the right the greater keys. The function also returns the counters used in the lecture notes: the number of key comparisons and the number of shifts (no swaps: a shift is a single assignment).

In [1]:
%%writefile insertion_sort.cpp
#include <cstdarg>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>
using namespace std;

// Helper functions to print the trace
string S(int x) { return to_string(x); }

string Str(int n, const int *a)     // the array as "[5, 2, 4, 1, 3]"
{
    string s = "[";
    for (int i = 0; i < n; i++) s += (i ? ", " : "") + to_string(a[i]);
    return s + "]";
}

string F(const char *fmt, ...)      // like printf, but returns a string
{
    char buf[200];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return buf;
}

void Row(string i, string j, string key, int n, const int *a, string action)    // one row of the trace
{
    printf("%3s %3s %4s  %-20s  %s\n", i.c_str(), j.c_str(), key.c_str(), Str(n, a).c_str(), action.c_str());
}

// Sorts a[0..n-1] in place; returns key comparisons and shifts
void InsertionSort(int n, int *a, int &comparisons, int &shifts, bool trace = false)
{
    int i, j, key;
    comparisons = 0;                    // key comparisons
    shifts = 0;                         // shifts
    if (trace)
        Row("-", "-", "-", n, a, "initial state");

    for (i = 1; i < n; i++)             // i = 2..n in the pseudo-code
    {
        key = a[i];                     // element to insert
        j = i - 1;
        if (trace)
            Row(S(i), S(j), S(key), n, a, F("key <- a[%d]=%d; j <- %d", i, key, j));

        // shift the keys greater than key to the right
        while (j >= 0 && a[j] > key)
        {
            comparisons++;
            shifts++;
            a[j + 1] = a[j];
            j = j - 1;
            if (trace)
                Row("", S(j), S(key), n, a, F("a[%d]=%d>%d: shift a[%d] <- %d; j <- %d", j + 1, a[j + 1], key, j + 2, a[j + 1], j));
        }
        if (j >= 0)
            comparisons++;              // the comparison that stopped the loop (a[j] <= key)

        // insert key in the correct position
        a[j + 1] = key;
        if (trace)
        {
            if (j < 0)
                Row("", "-", S(key), n, a, F("j=%d: exit; insert a[%d] <- %d", j, j + 1, key));
            else
                Row("", "-", S(key), n, a, F("a[%d]=%d<=%d: exit; insert a[%d] <- %d", j, a[j], key, j + 1, key));
        }
    }
}

void Run(string title, vector<int> v)
{
    int n = v.size(), comparisons, shifts;
    printf("%s: a = %s, n = %d\n", title.c_str(), Str(n, v.data()).c_str(), n);
    printf("%3s %3s %4s  %-20s  %s\n", "i", "j", "key", "a", "action");
    InsertionSort(n, v.data(), comparisons, shifts, true);
    printf("Sorted array: %s\n", Str(n, v.data()).c_str());
    printf("Key comparisons: %d, shifts: %d\n\n", comparisons, shifts);
}

int main(int argc, char *argv[])
{
    if (argc > 1)   // “Your turn”: the array comes from the command line
    {
        vector<int> v;
        for (int k = 1; k < argc; k++) v.push_back(atoi(argv[k]));
        Run("Your turn", v);
        return 0;
    }
    Run("Instance 1", {5, 2, 4, 1, 3});
    Run("Instance 2", {12, 11, 13, 5, 6});
    return 0;
}


Writing insertion_sort.cpp


### Execution on the examples of the chapter

In [2]:
!g++ -std=c++17 -O2 -o insertion_sort insertion_sort.cpp && ./insertion_sort

Instance 1: a = [5, 2, 4, 1, 3], n = 5
  i   j  key  a                     action
  -   -    -  [5, 2, 4, 1, 3]       initial state
  1   0    2  [5, 2, 4, 1, 3]       key <- a[1]=2; j <- 0
     -1    2  [5, 5, 4, 1, 3]       a[0]=5>2: shift a[1] <- 5; j <- -1
      -    2  [2, 5, 4, 1, 3]       j=-1: exit; insert a[0] <- 2
  2   1    4  [2, 5, 4, 1, 3]       key <- a[2]=4; j <- 1
      0    4  [2, 5, 5, 1, 3]       a[1]=5>4: shift a[2] <- 5; j <- 0
      -    4  [2, 4, 5, 1, 3]       a[0]=2<=4: exit; insert a[1] <- 4
  3   2    1  [2, 4, 5, 1, 3]       key <- a[3]=1; j <- 2
      1    1  [2, 4, 5, 5, 3]       a[2]=5>1: shift a[3] <- 5; j <- 1
      0    1  [2, 4, 4, 5, 3]       a[1]=4>1: shift a[2] <- 4; j <- 0
     -1    1  [2, 2, 4, 5, 3]       a[0]=2>1: shift a[1] <- 2; j <- -1
      -    1  [1, 2, 4, 5, 3]       j=-1: exit; insert a[0] <- 1
  4   3    3  [1, 2, 4, 5, 3]       key <- a[4]=3; j <- 3
      2    3  [1, 2, 4, 5, 5]       a[3]=5>3: shift a[4] <- 5; j <- 2
      1    3  

### Try it yourself

Change the numbers below and run the cell again.

In [3]:
!./insertion_sort 8 3 5 1 9 2

Your turn: a = [8, 3, 5, 1, 9, 2], n = 6
  i   j  key  a                     action
  -   -    -  [8, 3, 5, 1, 9, 2]    initial state
  1   0    3  [8, 3, 5, 1, 9, 2]    key <- a[1]=3; j <- 0
     -1    3  [8, 8, 5, 1, 9, 2]    a[0]=8>3: shift a[1] <- 8; j <- -1
      -    3  [3, 8, 5, 1, 9, 2]    j=-1: exit; insert a[0] <- 3
  2   1    5  [3, 8, 5, 1, 9, 2]    key <- a[2]=5; j <- 1
      0    5  [3, 8, 8, 1, 9, 2]    a[1]=8>5: shift a[2] <- 8; j <- 0
      -    5  [3, 5, 8, 1, 9, 2]    a[0]=3<=5: exit; insert a[1] <- 5
  3   2    1  [3, 5, 8, 1, 9, 2]    key <- a[3]=1; j <- 2
      1    1  [3, 5, 8, 8, 9, 2]    a[2]=8>1: shift a[3] <- 8; j <- 1
      0    1  [3, 5, 5, 8, 9, 2]    a[1]=5>1: shift a[2] <- 5; j <- 0
     -1    1  [3, 3, 5, 8, 9, 2]    a[0]=3>1: shift a[1] <- 3; j <- -1
      -    1  [1, 3, 5, 8, 9, 2]    j=-1: exit; insert a[0] <- 1
  4   3    9  [1, 3, 5, 8, 9, 2]    key <- a[4]=9; j <- 3
      -    9  [1, 3, 5, 8, 9, 2]    a[3]=8<=9: exit; insert a[4] <- 9
  5   4    2

## ExchangeSort

**Input:** an array $a[1..n]$ of $n$ natural numbers. **Output:** the array $a$ sorted in non-decreasing order.

At step $i$ the element $a[i]$ is compared with every $a[j]$ with $j > i$, and the two elements are swapped whenever $a[i] > a[j]$: at the end of the step $a[i]$ is the minimum of $a[i..n]$. The function returns the number of key comparisons and the number of swaps.

In [4]:
%%writefile exchange_sort.cpp
#include <cstdarg>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>
using namespace std;

// Helper functions to print the trace
string S(int x) { return to_string(x); }

string Str(int n, const int *a)     // the array as "[5, 2, 4, 1, 3]"
{
    string s = "[";
    for (int i = 0; i < n; i++) s += (i ? ", " : "") + to_string(a[i]);
    return s + "]";
}

string F(const char *fmt, ...)      // like printf, but returns a string
{
    char buf[200];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return buf;
}

void Row(string i, string j, int n, const int *a, string action)    // one row of the trace
{
    printf("%3s %3s  %-20s  %s\n", i.c_str(), j.c_str(), Str(n, a).c_str(), action.c_str());
}

// Sorts a[0..n-1] in place; returns key comparisons and swaps
void ExchangeSort(int n, int *a, int &comparisons, int &swaps, bool trace = false)
{
    comparisons = 0;                    // key comparisons
    swaps = 0;                          // swaps
    if (trace)
        Row("-", "-", n, a, "initial state");

    for (int i = 0; i < n - 1; i++)     // i = 1..n-1 in the pseudo-code
    {
        for (int j = i + 1; j < n; j++) // j = i+1..n in the pseudo-code
        {
            comparisons++;
            int ai = a[i], aj = a[j];   // values before the swap, for the trace
            if (a[i] > a[j])
            {
                int tmp = a[i];         // swap a[i] and a[j]
                a[i] = a[j];
                a[j] = tmp;
                swaps++;
                if (trace)
                    Row(j == i + 1 ? S(i) : "", S(j), n, a, F("a[%d]=%d>a[%d]=%d: swap a[%d] <-> a[%d]", i, ai, j, aj, i, j));
            }
            else if (trace)
                Row(j == i + 1 ? S(i) : "", S(j), n, a, F("a[%d]=%d<=a[%d]=%d: no swap", i, ai, j, aj));
        }
    }
}

void Run(string title, vector<int> v)
{
    int n = v.size(), comparisons, swaps;
    printf("%s: a = %s, n = %d\n", title.c_str(), Str(n, v.data()).c_str(), n);
    printf("%3s %3s  %-20s  %s\n", "i", "j", "a", "action");
    ExchangeSort(n, v.data(), comparisons, swaps, true);
    printf("Sorted array: %s\n", Str(n, v.data()).c_str());
    printf("Key comparisons: %d, swaps: %d\n\n", comparisons, swaps);
}

int main(int argc, char *argv[])
{
    if (argc > 1)   // “Your turn”: the array comes from the command line
    {
        vector<int> v;
        for (int k = 1; k < argc; k++) v.push_back(atoi(argv[k]));
        Run("Your turn", v);
        return 0;
    }
    Run("Instance 1", {5, 2, 4, 1, 3});
    Run("Instance 2", {12, 11, 13, 5, 6});
    return 0;
}


Writing exchange_sort.cpp


### Execution on the examples of the chapter

In [5]:
!g++ -std=c++17 -O2 -o exchange_sort exchange_sort.cpp && ./exchange_sort

Instance 1: a = [5, 2, 4, 1, 3], n = 5
  i   j  a                     action
  -   -  [5, 2, 4, 1, 3]       initial state
  0   1  [2, 5, 4, 1, 3]       a[0]=5>a[1]=2: swap a[0] <-> a[1]
      2  [2, 5, 4, 1, 3]       a[0]=2<=a[2]=4: no swap
      3  [1, 5, 4, 2, 3]       a[0]=2>a[3]=1: swap a[0] <-> a[3]
      4  [1, 5, 4, 2, 3]       a[0]=1<=a[4]=3: no swap
  1   2  [1, 4, 5, 2, 3]       a[1]=5>a[2]=4: swap a[1] <-> a[2]
      3  [1, 2, 5, 4, 3]       a[1]=4>a[3]=2: swap a[1] <-> a[3]
      4  [1, 2, 5, 4, 3]       a[1]=2<=a[4]=3: no swap
  2   3  [1, 2, 4, 5, 3]       a[2]=5>a[3]=4: swap a[2] <-> a[3]
      4  [1, 2, 3, 5, 4]       a[2]=4>a[4]=3: swap a[2] <-> a[4]
  3   4  [1, 2, 3, 4, 5]       a[3]=5>a[4]=4: swap a[3] <-> a[4]
Sorted array: [1, 2, 3, 4, 5]
Key comparisons: 10, swaps: 7

Instance 2: a = [12, 11, 13, 5, 6], n = 5
  i   j  a                     action
  -   -  [12, 11, 13, 5, 6]    initial state
  0   1  [11, 12, 13, 5, 6]    a[0]=12>a[1]=11: swap a[0] <-> a[1]
     

### Try it yourself

Change the numbers below and run the cell again.

In [6]:
!./exchange_sort 8 3 5 1 9 2

Your turn: a = [8, 3, 5, 1, 9, 2], n = 6
  i   j  a                     action
  -   -  [8, 3, 5, 1, 9, 2]    initial state
  0   1  [3, 8, 5, 1, 9, 2]    a[0]=8>a[1]=3: swap a[0] <-> a[1]
      2  [3, 8, 5, 1, 9, 2]    a[0]=3<=a[2]=5: no swap
      3  [1, 8, 5, 3, 9, 2]    a[0]=3>a[3]=1: swap a[0] <-> a[3]
      4  [1, 8, 5, 3, 9, 2]    a[0]=1<=a[4]=9: no swap
      5  [1, 8, 5, 3, 9, 2]    a[0]=1<=a[5]=2: no swap
  1   2  [1, 5, 8, 3, 9, 2]    a[1]=8>a[2]=5: swap a[1] <-> a[2]
      3  [1, 3, 8, 5, 9, 2]    a[1]=5>a[3]=3: swap a[1] <-> a[3]
      4  [1, 3, 8, 5, 9, 2]    a[1]=3<=a[4]=9: no swap
      5  [1, 2, 8, 5, 9, 3]    a[1]=3>a[5]=2: swap a[1] <-> a[5]
  2   3  [1, 2, 5, 8, 9, 3]    a[2]=8>a[3]=5: swap a[2] <-> a[3]
      4  [1, 2, 5, 8, 9, 3]    a[2]=5<=a[4]=9: no swap
      5  [1, 2, 3, 8, 9, 5]    a[2]=5>a[5]=3: swap a[2] <-> a[5]
  3   4  [1, 2, 3, 8, 9, 5]    a[3]=8<=a[4]=9: no swap
      5  [1, 2, 3, 5, 9, 8]    a[3]=8>a[5]=5: swap a[3] <-> a[5]
  4   5  [1, 2, 3, 5, 8, 9

## SelectionSort

**Input:** an array $a[1..n]$ of $n$ natural numbers. **Output:** the array $a$ sorted in non-decreasing order.

At step $i$ the inner loop looks for the index $\mathit{minIdx}$ of the minimum of $a[i..n]$ (the comparison is strict: in case of ties the first minimum wins) and then, if $\mathit{minIdx} \ne i$, swaps $a[i]$ with $a[\mathit{minIdx}]$: at most one swap per step. The function returns the number of key comparisons and the number of swaps.

In [7]:
%%writefile selection_sort.cpp
#include <cstdarg>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>
using namespace std;

// Helper functions to print the trace
string S(int x) { return to_string(x); }

string Str(int n, const int *a)     // the array as "[5, 2, 4, 1, 3]"
{
    string s = "[";
    for (int i = 0; i < n; i++) s += (i ? ", " : "") + to_string(a[i]);
    return s + "]";
}

string F(const char *fmt, ...)      // like printf, but returns a string
{
    char buf[200];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return buf;
}

void Row(string i, string j, string minIdx, int n, const int *a, string action)    // one row of the trace
{
    printf("%3s %3s %6s  %-20s  %s\n", i.c_str(), j.c_str(), minIdx.c_str(), Str(n, a).c_str(), action.c_str());
}

// Sorts a[0..n-1] in place; returns key comparisons and swaps
void SelectionSort(int n, int *a, int &comparisons, int &swaps, bool trace = false)
{
    comparisons = 0;                    // key comparisons
    swaps = 0;                          // swaps
    if (trace)
        Row("-", "-", "-", n, a, "initial state");

    for (int i = 0; i < n - 1; i++)     // i = 1..n-1 in the pseudo-code
    {
        int minIdx = i;                 // index of the minimum in a[i..n]
        if (trace)
            Row(S(i), "-", S(minIdx), n, a, F("minIdx := %d (a[%d]=%d)", i, i, a[i]));
        for (int j = i + 1; j < n; j++) // j = i+1..n in the pseudo-code
        {
            comparisons++;
            if (a[j] < a[minIdx])
            {
                int old = minIdx;       // only for the trace
                minIdx = j;             // new minimum found
                if (trace)
                    Row("", S(j), S(minIdx), n, a, F("a[%d]=%d<a[%d]=%d: minIdx <- %d", j, a[j], old, a[old], j));
            }
            else if (trace)
                Row("", S(j), S(minIdx), n, a, F("a[%d]=%d>=a[%d]=%d: no update", j, a[j], minIdx, a[minIdx]));
        }
        if (minIdx != i)
        {
            int tmp = a[i];             // swap a[i] and a[minIdx]
            a[i] = a[minIdx];
            a[minIdx] = tmp;
            swaps++;
            if (trace)
                Row("", "-", "-", n, a, F("minIdx=%d != %d: swap a[%d] <-> a[%d]", minIdx, i, i, minIdx));
        }
        else if (trace)
            Row("", "-", "-", n, a, F("minIdx=%d: no swap", minIdx));
    }
}

void Run(string title, vector<int> v)
{
    int n = v.size(), comparisons, swaps;
    printf("%s: a = %s, n = %d\n", title.c_str(), Str(n, v.data()).c_str(), n);
    printf("%3s %3s %6s  %-20s  %s\n", "i", "j", "minIdx", "a", "action");
    SelectionSort(n, v.data(), comparisons, swaps, true);
    printf("Sorted array: %s\n", Str(n, v.data()).c_str());
    printf("Key comparisons: %d, swaps: %d\n\n", comparisons, swaps);
}

int main(int argc, char *argv[])
{
    if (argc > 1)   // “Your turn”: the array comes from the command line
    {
        vector<int> v;
        for (int k = 1; k < argc; k++) v.push_back(atoi(argv[k]));
        Run("Your turn", v);
        return 0;
    }
    Run("Instance 1", {5, 2, 4, 1, 3});
    Run("Instance 2", {12, 11, 13, 5, 6});
    return 0;
}


Writing selection_sort.cpp


### Execution on the examples of the chapter

In [8]:
!g++ -std=c++17 -O2 -o selection_sort selection_sort.cpp && ./selection_sort

Instance 1: a = [5, 2, 4, 1, 3], n = 5
  i   j minIdx  a                     action
  -   -      -  [5, 2, 4, 1, 3]       initial state
  0   -      0  [5, 2, 4, 1, 3]       minIdx := 0 (a[0]=5)
      1      1  [5, 2, 4, 1, 3]       a[1]=2<a[0]=5: minIdx <- 1
      2      1  [5, 2, 4, 1, 3]       a[2]=4>=a[1]=2: no update
      3      3  [5, 2, 4, 1, 3]       a[3]=1<a[1]=2: minIdx <- 3
      4      3  [5, 2, 4, 1, 3]       a[4]=3>=a[3]=1: no update
      -      -  [1, 2, 4, 5, 3]       minIdx=3 != 0: swap a[0] <-> a[3]
  1   -      1  [1, 2, 4, 5, 3]       minIdx := 1 (a[1]=2)
      2      1  [1, 2, 4, 5, 3]       a[2]=4>=a[1]=2: no update
      3      1  [1, 2, 4, 5, 3]       a[3]=5>=a[1]=2: no update
      4      1  [1, 2, 4, 5, 3]       a[4]=3>=a[1]=2: no update
      -      -  [1, 2, 4, 5, 3]       minIdx=1: no swap
  2   -      2  [1, 2, 4, 5, 3]       minIdx := 2 (a[2]=4)
      3      2  [1, 2, 4, 5, 3]       a[3]=5>=a[2]=4: no update
      4      4  [1, 2, 4, 5, 3]       a[4]=3<

### Try it yourself

Change the numbers below and run the cell again.

In [9]:
!./selection_sort 8 3 5 1 9 2

Your turn: a = [8, 3, 5, 1, 9, 2], n = 6
  i   j minIdx  a                     action
  -   -      -  [8, 3, 5, 1, 9, 2]    initial state
  0   -      0  [8, 3, 5, 1, 9, 2]    minIdx := 0 (a[0]=8)
      1      1  [8, 3, 5, 1, 9, 2]    a[1]=3<a[0]=8: minIdx <- 1
      2      1  [8, 3, 5, 1, 9, 2]    a[2]=5>=a[1]=3: no update
      3      3  [8, 3, 5, 1, 9, 2]    a[3]=1<a[1]=3: minIdx <- 3
      4      3  [8, 3, 5, 1, 9, 2]    a[4]=9>=a[3]=1: no update
      5      3  [8, 3, 5, 1, 9, 2]    a[5]=2>=a[3]=1: no update
      -      -  [1, 3, 5, 8, 9, 2]    minIdx=3 != 0: swap a[0] <-> a[3]
  1   -      1  [1, 3, 5, 8, 9, 2]    minIdx := 1 (a[1]=3)
      2      1  [1, 3, 5, 8, 9, 2]    a[2]=5>=a[1]=3: no update
      3      1  [1, 3, 5, 8, 9, 2]    a[3]=8>=a[1]=3: no update
      4      1  [1, 3, 5, 8, 9, 2]    a[4]=9>=a[1]=3: no update
      5      5  [1, 3, 5, 8, 9, 2]    a[5]=2<a[1]=3: minIdx <- 5
      -      -  [1, 2, 5, 8, 9, 3]    minIdx=5 != 1: swap a[1] <-> a[5]
  2   -      2  [1, 2

### Comparison of the three algorithms

On the same two arrays, the three algorithms perform (as in the comparison example of the lecture notes):

| algorithm | $[5, 2, 4, 1, 3]$: comparisons | moves | $[12, 11, 13, 5, 6]$: comparisons | moves |
|---|---|---|---|---|
| Insertion Sort | 9 | 7 shifts | 9 | 7 shifts |
| Exchange Sort | 10 | 7 swaps | 10 | 7 swaps |
| Selection Sort | 10 | 3 swaps | 10 | 3 swaps |

Selection Sort always performs $n(n-1)/2$ comparisons but at most $n-1$ swaps; Insertion Sort adapts to the input (on an already sorted array it performs only $n-1$ comparisons and no shift) and is the only stable one of the three.

---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).